# 115 — Memoria, contexto y continuidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

El LLM es una función sin estado: solo "recuerda" su ventana de contexto. La
continuidad se construye con tres almacenes:

- **Contexto (memoria de trabajo):** instrucciones + plan + ternas recientes. Volátil,
  cara (se paga por token en CADA llamada), limitada.
- **Memoria persistente:** *episódica* (qué pasó: trazas) y *semántica* (hechos
  destilados con procedencia). Entra al contexto por recuperación selectiva.
- **Checkpoint:** instantánea del estado del bucle (plan + variables + log de efectos
  aplicados) en un punto consistente; su contrato es reanudar SIN repetir efectos.

### 📉 Gestionar el contexto que crece

Estrategias: **compactar** (ternas viejas → resumen estructurado; con pérdida — errores
y decisiones se conservan textuales), **externalizar** (detalle → archivo/BD + puntero
en contexto) y **seleccionar** (recuperar solo lo relevante al paso actual, parte 08).

```text
contexto = instrucciones (fijo) + plan (siempre visible)
         + resumen de lo hecho + últimas K ternas + recuperado bajo demanda
```

Riesgo propio de la memoria: lo persistido vuelve a entrar en sesiones futuras — un
dato envenenado hoy es una "verdad recordada" mañana. Etiquetar procedencia y curar
(caducidad, corrección) no es opcional.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Checkpoint válido tras el paso 1: plan = [verificar estado: HECHA,
sumar: PENDIENTE]; verificado = {"healthy": true} (anclado a la observación); log de
efectos = [status] (aunque sea pura, registrar evita re-ejecutar). Reanudar = cargar el
checkpoint, reconstruir contexto (instrucciones + plan + verificado) y ejecutar la
única sub-tarea PENDIENTE: `sum(7,5)`. El punto es consistente porque se tomó ENTRE
acciones, nunca en mitad de una.

**Ejercicio 2.** (a) Contexto en el paso n = 900 + 350 + 280n ≤ 6000 → n ≤ 16,9: la
llamada del paso 17 desborda. (b) Estable = 900 + 350 + 500 + 4×280 = 2.870 tokens.
Acumulado en 30 pasos: sin gestión ≈ Σ(1250 + 280n) = 37.500 + 280×465 ≈ 167.700
tokens de entrada (y además imposible desde el 17); con gestión ≈ 30 × 2.870 = 86.100.
La gestión no solo evita el muro: reduce el costo total aunque cada resumen cueste
tokens extra de generación.

**Ejercicio 3.** (a) semántica, procedencia usuario — preferencia estable; (b)
episódica, procedencia tool/traza — consultable, no cargarla entera al contexto; (c)
semántica, procedencia tool (medido) — hecho accionable con fecha de caducidad; (d) NO
persistir — secreto: vive solo en la configuración del runtime; (e) contexto de la
sesión (o semántica marcada como HIPÓTESIS, procedencia deducción) — persistirla como
hecho la convierte en falsa verdad recordada.

**Ejercicio 4.** Resumen válido (≤ 40 palabras): "Objetivo: estado + suma. Paso 1:
`status()` → healthy: true (verificado). Paso 2: `sum(7,5)` → 12 (verificado). Ambas
condiciones cumplidas; sin errores; sin efectos de escritura. Pendiente: nada." Se
sacrifican los JSON literales de las observaciones; si la tarea siguiera y un paso 25
necesitara el campo `service` de la observación 1, habría que recuperarlo de la
memoria episódica — por eso el detalle se externaliza, no se destruye.

In [ ]:
result = run_lab("agent", seed=115)
assert result["kind"] == "agent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — checkpoint y reanudación simulada
result = run_lab("agent", seed=115)
trace = result["result"]["trace"]

checkpoint_1 = {
    "plan": [
        {"subtarea": "verificar estado", "estado": "HECHA"},
        {"subtarea": "sumar 7+5", "estado": "PENDIENTE"},
    ],
    "verificado": {"healthy": trace[0]["observation"]["healthy"]},
    "log_efectos": [trace[0]["action"]["tool"]],
}

pendientes = [t["subtarea"] for t in checkpoint_1["plan"] if t["estado"] == "PENDIENTE"]
assert pendientes == ["sumar 7+5"]
assert "status" in checkpoint_1["log_efectos"]  # no se repite al reanudar
print("reanudar ejecutaria solo:", pendientes)


In [ ]:
# Ejercicio 2 — presupuesto de contexto, calculado
INSTRUCCIONES, PLAN, TERNA, VENTANA = 900, 350, 280, 6000

paso_desborde = next(n for n in range(1, 100)
                     if INSTRUCCIONES + PLAN + TERNA * n > VENTANA)
print("desborda en el paso:", paso_desborde)          # 17

contexto_estable = INSTRUCCIONES + PLAN + 500 + 4 * TERNA
print("contexto estable:", contexto_estable)          # 2870

acumulado_sin = sum(INSTRUCCIONES + PLAN + TERNA * n for n in range(1, 31))
acumulado_con = 30 * contexto_estable
print(f"acumulado sin gestion: {acumulado_sin:,} tokens (e imposible desde el paso 17)")
print(f"acumulado con gestion: {acumulado_con:,} tokens")


## Reflexión

1. El laboratorio termina en dos pasos y no necesita checkpoint. ¿En qué momento exacto
   de una tarea de 40 pasos tomarías checkpoints y por qué "nunca en mitad de un
   efecto"?
2. La compactación es una operación con pérdida. ¿Qué dos tipos de contenido deben
   conservarse textuales aunque todo lo demás se resuma, y qué error induce omitirlos?
3. ¿Por qué la memoria semántica recuperada debe tratarse como dato de baja procedencia
   aunque la haya escrito el propio agente?